# 05 — Forecast, calibration, and basis-risk evaluation

## tl;dr

Independently recompute the AQUASURE benchmark from 240,000 saved scenario rows. This notebook reports ranking, probability quality, trigger selectivity, negative basis risk, and pricing reconciliation.


## Context & Methods

- **ROC AUC:** Day-60 discrimination of severe versus non-severe outcomes.
- **Brier / Log Loss / ECE:** Day-60 probability quality.
- **Negative basis risk:** `P(no payout trigger | severe biological event)`.
- **Reduction:** weather-index NBR minus AQUASURE NBR, in percentage points.
- **Trigger probability:** share of full-cycle scenarios meeting every guardrail.

### Key assumptions

This is a calibrated synthetic laboratory benchmark. The coarse weather comparator has zero severe-event recall here, explaining its 100% negative basis risk. Field performance cannot be inferred.


In [ ]:
from pathlib import Path
import json
import math
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
ARTIFACTS = ROOT / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)
SEED = 20260830
print(f"AQUASURE project root: {ROOT}")


## Data

### 1. Load and validate scenario grain


In [ ]:
reference = json.loads((ROOT / "BENCHMARK_REFERENCE.json").read_text())
scenarios = pd.read_csv(ARTIFACTS / "monte_carlo_scenarios.csv")
pricing = json.loads((ARTIFACTS / "pricing_results.json").read_text())
assert len(scenarios) == reference["total_scenarios"]
assert scenarios.scenario_id.is_unique
assert scenarios.isna().sum().sum() == 0
print(f"Validated {len(scenarios):,} unique, complete scenario rows.")


## Results

### 2. Recompute Day-60 metrics


In [ ]:
from scipy.stats import rankdata
y = scenarios.severe_event.to_numpy(dtype=int); p = np.clip(scenarios.phri_day60.to_numpy(), 1e-12, 1-1e-12)
rank = rankdata(p); n_positive, n_negative = y.sum(), len(y)-y.sum()
roc_auc = (rank[y == 1].sum()-n_positive*(n_positive+1)/2)/(n_positive*n_negative)
brier = np.mean((p-y)**2); log_loss = -np.mean(y*np.log(p)+(1-y)*np.log(1-p))
bins = np.linspace(0, 1, 11); bin_id = np.minimum(np.digitize(p, bins[1:-1]), 9); ece = 0.0
for index in range(10):
    mask = bin_id == index
    if mask.any(): ece += mask.mean()*abs(p[mask].mean()-y[mask].mean())
probability_metrics = {"day60_roc_auc": float(roc_auc), "brier_score": float(brier), "log_loss": float(log_loss), "ece_10_equal_width_bins": float(ece)}
print(pd.Series(probability_metrics).round(5).to_string())


### 3. Recompute trigger and basis-risk metrics


In [ ]:
trigger = scenarios.claim_trigger.to_numpy(dtype=int); weather_trigger = scenarios.weather_index_trigger.to_numpy(dtype=int); severe = y == 1
negative_basis_risk = lambda vector: float(np.mean(vector[severe] == 0))
aquasure_nbr = negative_basis_risk(trigger); weather_nbr = negative_basis_risk(weather_trigger); reduction_pp = 100*(weather_nbr-aquasure_nbr)
insurance_metrics = {"severe_event_rate": float(y.mean()), "cycle_trigger_probability": float(trigger.mean()), "aquasure_negative_basis_risk": aquasure_nbr, "weather_index_negative_basis_risk": weather_nbr, "basis_risk_reduction_percentage_points": reduction_pp}
print(pd.Series(insurance_metrics).round(5).to_string())


### 4. Reconcile observed values with recovered reference


In [ ]:
observed = {**probability_metrics, **insurance_metrics, "expected_payout_idr": float(scenarios.payout_idr.mean()), "wang_distortion_premium_idr": float(pricing["wang_distortion_premium_idr"]), "gross_premium_idr": float(pricing["gross_premium_idr"])}
rows = []
for metric, actual in observed.items():
    reference_key = "ece" if metric == "ece_10_equal_width_bins" else metric; target = reference.get(reference_key, np.nan)
    rows.append({"metric": metric, "observed": actual, "reference": target, "difference": actual-target})
comparison = pd.DataFrame(rows)
print(comparison.round(5).to_string(index=False))


### 5. Save judge-facing evidence


In [ ]:
evaluation = {"evidence_status": "calibrated synthetic reconstruction; not field validation", "scenario_population": "12 farms × 20,000 scenarios", "metric_definitions": {"negative_basis_risk": "P(no trigger | severe biological event)", "reduction": "weather-index NBR minus AQUASURE NBR, percentage points", "ece": "10 equal-width bins"}, "observed": observed, "reference": reference}
(ARTIFACTS / "evaluation_metrics.json").write_text(json.dumps(evaluation, indent=2))
comparison.to_csv(ARTIFACTS / "benchmark_reconciliation.csv", index=False)
print(f"\nHeadline: {reduction_pp:.2f} percentage-point reduction in negative basis risk.")


## Takeaways

The exact defensible claim is: **AQUASURE reduced negative basis risk by 57.14 percentage points relative to a coarse weather-index benchmark in calibrated synthetic evaluation.** Do not call it a 57.14% reduction in total basis risk or observed commercial performance.


## Limitations

Before TRL/TKT 5, replace synthetic inputs with real sensor cycles, survival/biomass/harvest outcomes, and insurer-reviewed claims. Refit biology, recalibrate PHRI, freeze model governance, and repeat out-of-time validation.
